# EcoSort AI — pipeline definitiva

Dal dataset grezzo al file `.tflite` pronto per il Raspberry Pi.

| passo | cosa fa | quanto dura |
|---|---|---|
| 0 | setup e caricamento file | 2 min |
| 1 | dedup + split stratificato | 3-5 min |
| 2 | benchmark di 4 backbone | 2-3 ore (riavviabile) |
| 3 | grafico accuratezza / latenza | 10 s |
| 4 | training finale su train+val | 30-45 min |
| 5 | conversione TFLite + verifica | 10 min |
| 6 | download del pacchetto per il Pi | 1 min |

**Runtime → Cambia tipo di runtime → GPU (T4)** prima di iniziare.

Il passo 2 e' lungo: lo stato viene salvato su Drive dopo ogni backbone, quindi
se la sessione cade basta rilanciare la stessa cella e riprende da dove era.

## Passo 0 — Setup

In [ ]:
import tensorflow as tf, sys
print("TensorFlow:", tf.__version__)
print("Python    :", sys.version.split()[0])
gpu = tf.config.list_physical_devices('GPU')
print("GPU       :", gpu if gpu else "NESSUNA — vai su Runtime > Cambia tipo di runtime > T4")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || true

In [ ]:
# Drive serve per non perdere il lavoro se la sessione cade
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ecosort
print("cartella di lavoro: /content/drive/MyDrive/ecosort")

### Carica i 5 moduli Python

`ecosort_decisione.py`, `prepara_dataset.py`, `ecosort_benchmark.py`,
`train_finale.py`, `converti_tflite.py`

(`classifica_pi.py` non serve qui: va solo sul Pi.)

In [ ]:
from google.colab import files
import os
os.chdir('/content')
attesi = ['ecosort_decisione.py','prepara_dataset.py','ecosort_benchmark.py',
          'train_finale.py','converti_tflite.py']
mancanti = [f for f in attesi if not os.path.exists(f)]
if mancanti:
    print("Carica:", mancanti)
    files.upload()
print("presenti:", [f for f in attesi if os.path.exists(f)])

### Il dataset

Deve essere in `/content/dataset/` con una sottocartella per classe:

```
/content/dataset/carta_e_cartone/
/content/dataset/plastica/
/content/dataset/vetro_e_metallo/
```

Se lo tieni zippato su Drive (consigliato: scompattare da Drive e' molto piu
veloce che ricaricarlo ogni volta), modifica il percorso qui sotto.

In [ ]:
ZIP_SU_DRIVE = '/content/drive/MyDrive/dataset_definitivo.zip'   # <-- il tuo zip

import os, shutil
EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def conta_img(d):
    try:
        return sum(1 for f in os.listdir(d) if f.lower().endswith(EXT))
    except Exception:
        return 0

# 1. scompatta (se non l'hai gia fatto)
if not any(os.path.isdir(f'/content/{n}') and n != 'drive' and conta_img(f'/content/{n}') == 0
           and os.listdir(f'/content/{n}') for n in os.listdir('/content')):
    pass
if os.path.exists(ZIP_SU_DRIVE):
    !unzip -q -o "$ZIP_SU_DRIVE" -d /content/
else:
    print(f"Non trovo {ZIP_SU_DRIVE} — correggi il percorso qui sopra")

# 2. mostra cosa e finito in /content
print("Contenuto di /content:")
for n in sorted(os.listdir('/content')):
    p = f'/content/{n}'
    if os.path.isdir(p) and n != 'drive':
        sub = sorted(s for s in os.listdir(p) if os.path.isdir(f'{p}/{s}'))
        print(f"  {n}/   sottocartelle: {sub[:6]}{' ...' if len(sub) > 6 else ''}")

# 3. trova la cartella che contiene davvero le classi.
#    Gli zip spesso hanno una cartella radice (dataset_definitivo/, o peggio
#    un doppio annidamento): invece di indovinare, la cerchiamo.
candidati = []
for radice, dirs, _ in os.walk('/content'):
    if '/drive' in radice or '/.' in radice:
        dirs[:] = []
        continue
    if radice[len('/content'):].count(os.sep) > 3:
        dirs[:] = []
        continue
    classi = sorted(d for d in dirs if conta_img(os.path.join(radice, d)) >= 10)
    if len(classi) >= 2:
        candidati.append((radice, classi, sum(conta_img(os.path.join(radice, c)) for c in classi)))

if not candidati:
    print("\nNessuna cartella di classi trovata: controlla il contenuto dello zip.")
else:
    candidati.sort(key=lambda x: -x[2])
    print("\nCandidati:")
    for r, c, n in candidati:
        print(f"  {r}  ->  {c}  ({n} immagini)")

    # 4. collega il migliore a /content/dataset, che e cio che si aspettano gli script
    sorgente, classi, tot = candidati[0]
    dest = '/content/dataset'
    if os.path.abspath(sorgente) != os.path.abspath(dest):
        if os.path.islink(dest):
            os.unlink(dest)
        elif os.path.isdir(dest):
            shutil.rmtree(dest)
        os.symlink(sorgente, dest)
        print(f"\ncollegato {sorgente} -> {dest}")

    print(f"\nDataset pronto: {tot} immagini")
    for c in sorted(os.listdir(dest)):
        if os.path.isdir(f'{dest}/{c}'):
            print(f"  {c:>20}: {conta_img(f'{dest}/{c}')}")

### (opzionale ma consigliato) Le foto scattate dal Pi

Se hai raccolto 200-250 foto con la camera dentro la scatola, caricale come
`foto_pi.zip` su Drive con la stessa struttura a cartelle. Diventeranno il test
set: misurano quanto il modello regge il **domain shift** fra le foto trovate
su internet e le condizioni reali della tua macchina.

Senza queste, il test set viene ritagliato dal dataset web e risponde a una
domanda diversa da quella che ti interessa.

In [ ]:
ZIP_FOTO_PI = '/content/drive/MyDrive/foto_pi.zip'   # <-- opzionale

import os
if os.path.exists(ZIP_FOTO_PI) and not os.path.isdir('/content/foto_pi'):
    !unzip -q "$ZIP_FOTO_PI" -d /content/
if os.path.isdir('/content/foto_pi'):
    tot = sum(len(os.listdir(f'/content/foto_pi/{c}')) for c in os.listdir('/content/foto_pi'))
    print(f"foto reali dal Pi: {tot} — saranno il test set")
else:
    print("nessuna foto dal Pi: userai --modo classico al passo 1")

## Passo 1 — Preparazione del dataset

Cerca i duplicati (md5 + hash percettivo) e crea gli split stratificati in
`split.json`. Da eseguire **una volta sola**.

- `--modo pi` → 80/20, il test sono le foto del Pi *(usa questo se le hai)*
- `--modo classico` → 70/15/15 tutto interno al dataset web

Il numero da guardare nell'output: **la percentuale di duplicati rimossi**.
Se supera il 10%, l'accuratezza che avevi prima era gonfiata.

In [ ]:
MODO = 'pi' if os.path.isdir('/content/foto_pi') else 'classico'
print("modo:", MODO)
!python prepara_dataset.py --dataset /content/dataset --modo $MODO --out /content/split.json

## Passo 2 — Benchmark dei backbone

Addestra 4 backbone con la stessa identica pipeline e li confronta sul **costo
degli errori**, non sull'accuratezza.

Ogni backbone impiega 25-40 minuti su T4. Lo stato viene salvato su Drive dopo
ognuno: **se la sessione cade, rilancia questa stessa cella** e riprende dal
backbone successivo.

Per saltarne qualcuno, modifica `DA_TESTARE` in `ecosort_benchmark.py`.

In [ ]:
!cp /content/split.json /content/split.json 2>/dev/null
!python ecosort_benchmark.py

## Passo 3 — Accuratezza contro latenza

Il vincitore non e' il modello piu accurato: e' quello che minimizza il costo
restando dentro il budget di 2-3 secondi sul Pi.

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt

BG, ACC, MUTED, TXT = '#0A0F0D', '#D4FF3A', '#9CA3AF', '#E8EFE9'
ris = json.load(open('/content/drive/MyDrive/ecosort/stato_benchmark.json'))['risultati']
ris = sorted(ris, key=lambda r: min(r['costo_decisione'], r['costo_soglia70']))
vincitore = ris[0]['backbone']

fig, ax = plt.subplots(figsize=(9, 5.5))
fig.patch.set_facecolor(BG); ax.set_facecolor(BG)

# stima del tempo sul Pi 4B: ~4.5x il tempo su CPU x86 di Colab
FATTORE_PI = 4.5
ax.axvspan(3000, 12000, color=MUTED, alpha=0.10)
ax.axvline(3000, color=MUTED, ls='--', lw=1.2)
ax.text(3100, ax.get_ylim()[0], ' fuori budget', color=MUTED, fontsize=9, va='bottom')

for r in ris:
    x = r['latenza_cpu_ms'] * FATTORE_PI
    y = r['accuracy'] * 100
    vince = r['backbone'] == vincitore
    ax.scatter(x, y, s=260 if vince else 110,
               color=ACC if vince else MUTED,
               edgecolor=BG, linewidth=2, zorder=3)
    etichetta = f"{r['backbone']}\ncosto {r['costo_decisione']:.3f}"
    if vince:
        etichetta += "  ← SCELTO"
    ax.annotate(etichetta, (x, y), textcoords='offset points', xytext=(12, -4),
                color=ACC if vince else TXT, fontsize=9,
                fontweight='bold' if vince else 'normal')

ax.set_xlabel('latenza stimata sul Pi 4B (ms)', color=TXT)
ax.set_ylabel('accuratezza sul test set (%)', color=TXT)
ax.set_title('Ogni punto e un backbone — il colore indica quello scelto',
             color=ACC, fontweight='bold', loc='left')
ax.set_xscale('log')
ax.grid(alpha=0.12, color=MUTED)
ax.tick_params(colors=TXT)
for s in ax.spines.values():
    s.set_color(MUTED)
plt.tight_layout(); plt.show()

print(f"{'backbone':<20}{'acc':>8}{'costo':>9}{'gravi':>7}{'ms x86':>9}{'ms Pi~':>9}")
for r in ris:
    print(f"{r['backbone']:<20}{r['accuracy']*100:>7.2f}%{r['costo_decisione']:>9.4f}"
          f"{r['gravi_decisione']:>7}{r['latenza_cpu_ms']:>9.0f}{r['latenza_cpu_ms']*FATTORE_PI:>9.0f}")

## Passo 4 — Training finale

La validation ha finito il suo lavoro (scegliere backbone ed epoche). Ora si
riaddestra il vincitore su **train + validation uniti**: circa il 25% di dati in
piu. Il test set resta intoccato.

Niente early stopping qui — si usano le epoche che il benchmark ha individuato.

In [ ]:
!python train_finale.py

## Passo 5 — Conversione in TFLite

Converte in tre varianti (float32, float16, INT8) e **le valuta tutte sul test
set** prima di scegliere. La quantizzazione puo' costare punti di accuratezza e
l'unico modo per saperlo e' misurarla.

Due cose importanti che succedono qui:

- **il preprocessing entra dentro il modello** — il `.tflite` accetta
  direttamente l'immagine della camera, quindi e' impossibile che il
  preprocessing sul Pi diverga da quello del training
- **l'output resta float32** anche nella versione INT8, perche' le probabilita'
  servono alla regola di decisione a costo

In [ ]:
!python converti_tflite.py

## Passo 6 — Pacchetto per il Raspberry Pi

In [ ]:
import shutil, os
os.makedirs('/content/pacchetto_pi', exist_ok=True)
OUT = '/content/drive/MyDrive/ecosort'
for f in ['rifiuti.tflite', 'config.json']:
    shutil.copy(f'{OUT}/{f}', '/content/pacchetto_pi/')
shutil.copy('/content/ecosort_decisione.py', '/content/pacchetto_pi/')
# classifica_pi.py: caricalo tu se non e gia in /content
if os.path.exists('/content/classifica_pi.py'):
    shutil.copy('/content/classifica_pi.py', '/content/pacchetto_pi/')

shutil.make_archive('/content/ecosort_pi', 'zip', '/content/pacchetto_pi')
shutil.copy('/content/ecosort_pi.zip', f'{OUT}/ecosort_pi.zip')
print(json.dumps(json.load(open(f'{OUT}/config.json')), indent=2)[:900])

from google.colab import files
files.download('/content/ecosort_pi.zip')

## Sul Raspberry Pi

```bash
# ambiente virtuale NON attivo
pip install ai-edge-litert numpy pillow --break-system-packages

unzip ecosort_pi.zip -d ~/ecosort && cd ~/ecosort

python3 classifica_pi.py --bench            # latenza reale: il numero vero
python3 classifica_pi.py --foto prova.jpg   # su un file
python3 classifica_pi.py                    # INVIO per scattare
```

**Non** usare `pip install tflite-runtime`: quel pacchetto e' fermo alla 2.14.0
di ottobre 2023 e ha wheel solo fino a Python 3.11, quindi sul tuo Pi con
Python 3.13 non si installa. Il successore e' `ai-edge-litert`.

Se `--bench` dovesse superare i 2 secondi al p95, la strada piu rapida e'
rifare il passo 5 forzando `scelta = 'int8'` in `converti_tflite.py`, oppure
rilanciare il benchmark con `DA_TESTARE = ['MobileNetV3Large']`.